<a href="https://colab.research.google.com/github/keshav123333/Pytorch_beginners/blob/main/amazonml2026/amazonml2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 32.9 MB/s eta 0:00:00


In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
s2=pd.read_csv("/content/drive/MyDrive/amazonml/train_source2.tsv",sep="\t")
s3=pd.read_csv("/content/drive/MyDrive/amazonml/train_source3.tsv",sep="\t")

In [ ]:
s1=pd.read_csv("/content/drive/MyDrive/amazonml/train_source1.tsv",sep="\t")

In [ ]:
truth=pd.read_csv("/content/drive/MyDrive/amazonml/train_ground_truth.tsv",sep="\t")
truth=truth.head(100)

In [ ]:
s1=s1.reset_index(drop=True)

In [ ]:

s2 = s2.reset_index(drop=True)
s3 = s3.reset_index(drop=True)

In [ ]:
s1.set_index("entity_id",inplace=True)
s2.set_index("entity_id",inplace=True)
s3.set_index("entity_id",inplace=True)

In [ ]:
cols = ["business_name", "business_address", "country"]

s1["text"] = (
    s1["business_name"].fillna("").astype(str) + " " +
    s1["business_address"].fillna("").astype(str) + " " +
    s1["country"].fillna("").astype(str)
)

s2["text"] = (
    s2["business_name"].fillna("").astype(str) + " " +
    s2["business_address"].fillna("").astype(str) + " " +
    s2["country"].fillna("").astype(str)
)

s3["text"] = (
    s3["business_name"].fillna("").astype(str) + " " +
    s3["business_address"].fillna("").astype(str) + " " +
    s3["country"].fillna("").astype(str)
)

In [ ]:
def make_text(row):
  return f"{row["business_name"]} {row["business_address"]} {row["country"]}"

In [ ]:
s1["text"]=s1.apply(make_text,axis=1)

In [ ]:

s2["text"]=s2.apply(make_text,axis=1)
s3["text"]=s3.apply(make_text,axis=1)

In [ ]:
s1

,entity_id,business_name,business_address,country,text
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,"Orelee's Barbershop 1795 Westchester Drive, Hi..."
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,"Prime Money 17560 Ellis Road, Tahlequah, OK US"
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,"B+ Retail Inc 1712 Montebello Avenue, Phoenix,..."
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,"Christ Chapel 2100 Cameron Drive, Unit APARTME..."
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,"Prabhav Business Center 797, Lake Town Block A..."
...,...,...,...,...,...
2206816,S1-479751630,Arlyn Farooqi Prairie Medical,"16804 Stevenage Street, Surprise, AZ",US,Arlyn Farooqi Prairie Medical 16804 Stevenage ...
2206817,S1-392755726,Patriot Fund,"MD, 7709 Ashdale Road, Capitol Heights",US,"Patriot Fund MD, 7709 Ashdale Road, Capitol He..."
2206818,S1-716147977,Sloan Fact of Yonkers,"28 Armstrong Avenue, Unit Apartment 2, Yonkers...",US,"Sloan Fact of Yonkers 28 Armstrong Avenue, Uni..."
2206819,S1-770136229,Ritter's Machine Works,"5609 Spring Meadow Road, Austin, TX",US,Ritter's Machine Works 5609 Spring Meadow Road...


In [ ]:
s2["sources"]="s2"
s3["sources"]="s3"

In [ ]:
candidates=pd.concat([s2,s3],ignore_index=True)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
emb=model.encode(candidates["text"].tolist(),normalize_embeddings=True,show_progress_bar=True)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
# import faiss
# import numpy as np
# embeddings=np.asarray(emb,dtype="float32")
# dimension=embeddings.shape[1]
# index=faiss.IndexFlatIP(dimension)
# index.add(embeddings)
# print(index.ntotal)

200


In [ ]:
import faiss
import numpy as np
embeddings=np.asarray(emb,dtype="float32")
dimension=embeddings.shape[1]
index=faiss.IndexFlatIP(dimension)


res = faiss.StandardGpuResources()
gpu_index = faiss.index_cpu_to_gpu(res, 0, index)

# 3. Embeddings GPU index mein add karo
gpu_index.add(embeddings)



AttributeError: module 'faiss' has no attribute 'StandardGpuResources'

In [ ]:
# # ye graph wala
# import faiss
# import numpy as np
# embeddings=np.asarray(emb,dtype="float32")
# dimension=embeddings.shape[1]
# index=faiss.IndexHNSWFlat(dimension)


# res = faiss.StandardGpuResources()
# gpu_index = faiss.index_cpu_to_gpu(res, 0, index)

# # 3. Embeddings GPU index mein add karo
# gpu_index.add(embeddings)



TypeError: Wrong number or type of arguments for overloaded function 'new_IndexHNSWFlat'.
  Possible C/C++ prototypes are:
    faiss::IndexHNSWFlat::IndexHNSWFlat()
    faiss::IndexHNSWFlat::IndexHNSWFlat(int,int,faiss::MetricType)
    faiss::IndexHNSWFlat::IndexHNSWFlat(int,int)


In [ ]:
import faiss
import numpy as np

# upar wala   USE NI AS 1 CRORE DATSET TOH VO GPU KE SATH USE NI AND YAHA PE CLUSTER BANA KE SEARCH KAR RAHA MAINE CLUSTER BANTA THEN CETROID SE FIND KI
# SABSE PAASVALA CLUSTER KAUNSA AND LIKE THIS
# --------------------------------------------------
# 1. Embeddings
# --------------------------------------------------

embeddings = np.asarray(emb, dtype="float32")

dimension = embeddings.shape[1]

# IMPORTANT:
# Agar embeddings already normalized hain to dobara normalize
# karne ki zarurat nahi.
#
# Agar nahi hain:



# --------------------------------------------------
# 2. Create IVF index
# --------------------------------------------------

nlist = 4096

quantizer = faiss.IndexFlatIP(dimension)

index = faiss.IndexIVFFlat(
    quantizer,
    dimension,
    nlist,
    faiss.METRIC_INNER_PRODUCT
)


# --------------------------------------------------
# 3. Train IVF
# --------------------------------------------------

sample_size = min(300_000, len(embeddings))

rng = np.random.default_rng(42)

sample_idx = rng.choice(
    len(embeddings),
    size=sample_size,
    replace=False
)

print("Training IVF index...")

index.train(embeddings[sample_idx])

print("Training completed.")


# --------------------------------------------------
# 4. Add 1 crore embeddings in batches
# --------------------------------------------------

batch_size = 100_000

for start in range(0, len(embeddings), batch_size):

    end = min(
        start + batch_size,
        len(embeddings)
    )

    index.add(embeddings[start:end])

    print(
        f"Added {end:,}/{len(embeddings):,}"
    )


# --------------------------------------------------
# 5. Number of clusters searched per query
# --------------------------------------------------

index.nprobe = 50

print("Total vectors:", index.ntotal)
print("Index ready.")

Training IVF index...


RuntimeError: Error in void faiss::Clustering::train_encoded(faiss::idx_t, const uint8_t*, const faiss::Index*, faiss::Index&, const float*) at /project/faiss/Clustering.cpp:66: Error: 'nx >= static_cast<idx_t>(k)' failed: Number of training points (200) should be at least as large as number of clusters (4096)

In [ ]:
#  ye har version upar wale code ka indexing save karne ka
import faiss
import numpy as np
import os
import gc


# ============================================================
# 1. BASIC INFORMATION
# ============================================================

print("============================================================")
print("FAISS IVF INDEX CREATION")
print("============================================================")

print("S2 shape:", emb2.shape)
print("S3 shape:", emb3.shape)


dimension = emb2.shape[1]

s2_len = len(emb2)
s3_len = len(emb3)

total_vectors = s2_len + s3_len


print("\n==============================")
print("DATASET INFORMATION")
print("==============================")

print("S2 vectors   :", f"{s2_len:,}")
print("S3 vectors   :", f"{s3_len:,}")
print("Total vectors:", f"{total_vectors:,}")
print("Dimension    :", dimension)


# ============================================================
# 2. CHECK DIMENSIONS
# ============================================================

assert emb3.shape[1] == dimension, (
    f"Dimension mismatch: "
    f"S2={dimension}, S3={emb3.shape[1]}"
)


# ============================================================
# 3. CHECK DATA TYPE
# ============================================================

print("\n==============================")
print("DATA TYPE")
print("==============================")

print("emb2 dtype:", emb2.dtype)
print("emb3 dtype:", emb3.dtype)


# ============================================================
# 4. IVF PARAMETERS
# ============================================================
#
# 1 crore scale ke liye:
#
# nlist = 16384
#
# Higher nlist:
#   -> smaller inverted lists
#   -> faster search
#   -> potentially slightly more training cost
#
# ============================================================

nlist = 16384

print("\n==============================")
print("IVF PARAMETERS")
print("==============================")

print("nlist:", nlist)


# ============================================================
# 5. CREATE QUANTIZER
# ============================================================
#
# Embeddings normalized hain.
#
# Inner Product == Cosine Similarity
# for L2-normalized vectors.
#
# ============================================================

quantizer = faiss.IndexFlatIP(
    dimension
)


# ============================================================
# 6. CREATE IVF INDEX
# ============================================================

ivf_index = faiss.IndexIVFFlat(
    quantizer,
    dimension,
    nlist,
    faiss.METRIC_INNER_PRODUCT
)


# ============================================================
# 7. WRAP WITH IDMAP2
# ============================================================
#
# This allows us to manually assign IDs:
#
# S2:
#   0 ... s2_len-1
#
# S3:
#   s2_len ... s2_len+s3_len-1
#
# ============================================================

index = faiss.IndexIDMap2(
    ivf_index
)


print("\nIndex created.")


# ============================================================
# 8. TRAINING SAMPLE
# ============================================================
#
# We DO NOT train on all vectors.
#
# Maximum:
#   500,000 vectors
#
# Approximately:
#   250k from S2
#   250k from S3
#
# ============================================================

sample_size = min(
    500_000,
    total_vectors
)

rng = np.random.default_rng(42)


# ============================================================
# 9. SAMPLE S2
# ============================================================

sample_s2 = min(
    sample_size // 2,
    s2_len
)


# ============================================================
# 10. SAMPLE S3
# ============================================================

sample_s3 = min(
    sample_size - sample_s2,
    s3_len
)


print("\n==============================")
print("TRAINING SAMPLE")
print("==============================")

print(
    "S2 training samples:",
    f"{sample_s2:,}"
)

print(
    "S3 training samples:",
    f"{sample_s3:,}"
)

print(
    "Total training samples:",
    f"{sample_s2 + sample_s3:,}"
)


# ============================================================
# 11. RANDOM INDICES
# ============================================================

idx2 = rng.choice(
    s2_len,
    size=sample_s2,
    replace=False
)

idx3 = rng.choice(
    s3_len,
    size=sample_s3,
    replace=False
)


# ============================================================
# 12. CREATE TRAINING DATA
# ============================================================

train_data = np.concatenate(
    [
        np.asarray(
            emb2[idx2],
            dtype="float32"
        ),

        np.asarray(
            emb3[idx3],
            dtype="float32"
        )
    ],
    axis=0
)


print(
    "\nTraining data shape:",
    train_data.shape
)

print(
    "Training data dtype:",
    train_data.dtype
)


# ============================================================
# 13. TRAIN IVF
# ============================================================
#
# IMPORTANT:
#
# emb2 and emb3 are already normalized.
#
# Therefore:
#
# DO NOT call:
#
# faiss.normalize_L2(train_data)
#
# ============================================================

print("\n==============================")
print("TRAINING IVF")
print("==============================")

print("Training started...")

index.train(
    train_data
)

print("Training completed.")


# ============================================================
# 14. FREE TRAINING MEMORY
# ============================================================

del train_data
del idx2
del idx3

gc.collect()


print("\nTraining memory released.")


# ============================================================
# 15. CHECK TRAINING STATUS
# ============================================================

print("\n==============================")
print("INDEX STATUS")
print("==============================")

print(
    "Is trained:",
    index.is_trained
)

assert index.is_trained, (
    "FAISS index training failed."
)


# ============================================================
# 16. ADD S2
# ============================================================
#
# S2 IDs:
#
# 0
# 1
# 2
# ...
# s2_len - 1
#
# ============================================================

batch_size = 100_000


print("\n==============================")
print("ADDING S2")
print("==============================")

print(
    "Batch size:",
    f"{batch_size:,}"
)


for start in range(
    0,
    s2_len,
    batch_size
):

    end = min(
        start + batch_size,
        s2_len
    )


    # --------------------------------------------------------
    # Load current batch
    # --------------------------------------------------------

    batch = np.asarray(
        emb2[start:end],
        dtype="float32"
    ).copy()


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # emb2 is already normalized.
    #
    # DO NOT normalize again.
    #
    # --------------------------------------------------------


    # --------------------------------------------------------
    # Global S2 IDs
    # --------------------------------------------------------

    ids = np.arange(
        start,
        end,
        dtype=np.int64
    )


    # --------------------------------------------------------
    # Add to index
    # --------------------------------------------------------

    index.add_with_ids(
        batch,
        ids
    )


    # --------------------------------------------------------
    # Free batch memory
    # --------------------------------------------------------

    del batch
    del ids

    gc.collect()


    print(
        f"S2 added: {end:,}/{s2_len:,}"
    )


print("\nS2 addition completed.")


# ============================================================
# 17. ADD S3
# ============================================================
#
# S3 IDs start AFTER S2.
#
# Example:
#
# S2 = 5,000,000
#
# S2 IDs:
#   0 ... 4,999,999
#
# S3 IDs:
#   5,000,000 ...
#
# ============================================================

s3_offset = s2_len


print("\n==============================")
print("ADDING S3")
print("==============================")

print(
    "S3 ID offset:",
    f"{s3_offset:,}"
)


for start in range(
    0,
    s3_len,
    batch_size
):

    end = min(
        start + batch_size,
        s3_len
    )


    # --------------------------------------------------------
    # Load current batch
    # --------------------------------------------------------

    batch = np.asarray(
        emb3[start:end],
        dtype="float32"
    ).copy()


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # emb3 is already normalized.
    #
    # DO NOT normalize again.
    #
    # --------------------------------------------------------


    # --------------------------------------------------------
    # Global S3 IDs
    # --------------------------------------------------------

    ids = np.arange(
        s3_offset + start,
        s3_offset + end,
        dtype=np.int64
    )


    # --------------------------------------------------------
    # Add to index
    # --------------------------------------------------------

    index.add_with_ids(
        batch,
        ids
    )


    # --------------------------------------------------------
    # Free batch memory
    # --------------------------------------------------------

    del batch
    del ids

    gc.collect()


    print(
        f"S3 added: {end:,}/{s3_len:,}"
    )


print("\nS3 addition completed.")


# ============================================================
# 18. CHECK TOTAL VECTORS
# ============================================================

print("\n==============================")
print("VECTOR COUNT CHECK")
print("==============================")

print(
    "Expected vectors:",
    f"{total_vectors:,}"
)

print(
    "Indexed vectors:",
    f"{index.ntotal:,}"
)


assert index.ntotal == total_vectors, (
    f"Vector count mismatch! "
    f"Expected {total_vectors}, "
    f"got {index.ntotal}"
)


# ============================================================
# 19. IVF SEARCH PARAMETER
# ============================================================
#
# nprobe controls how many IVF clusters are searched.
#
# nprobe higher:
#   -> better recall
#   -> slower
#
# Starting value:
#   50
#
# ============================================================

ivf_index.nprobe = 50


print("\n==============================")
print("SEARCH PARAMETERS")
print("==============================")

print(
    "nlist :",
    ivf_index.nlist
)

print(
    "nprobe:",
    ivf_index.nprobe
)


# ============================================================
# 20. TEST SEARCH
# ============================================================
#
# Before saving the huge index,
# test whether search works correctly.
#
# Use first 10 S1 embeddings.
#
# Make sure emb1 already exists.
#
# ============================================================

print("\n==============================")
print("TEST SEARCH")
print("==============================")


test_size = min(
    10,
    len(emb1)
)

test_queries = np.asarray(
    emb1[:test_size],
    dtype="float32"
)


# ------------------------------------------------------------
# IMPORTANT:
#
# emb1 is also assumed to be already normalized.
#
# ------------------------------------------------------------

test_scores, test_ids = index.search(
    test_queries,
    12
)


print(
    "Query shape:",
    test_queries.shape
)

print(
    "Scores shape:",
    test_scores.shape
)

print(
    "IDs shape:",
    test_ids.shape
)


print("\nFirst query IDs:")

print(
    test_ids[0]
)


print("\nFirst query scores:")

print(
    test_scores[0]
)


# ============================================================
# 21. BASIC SEARCH VALIDATION
# ============================================================

assert test_scores.shape == (
    test_size,
    12
)

assert test_ids.shape == (
    test_size,
    12
)


print("\nTest search successful.")


# ============================================================
# 22. SAVE INDEX
# ============================================================

output_file = (
    "/kaggle/working/"
    "s2_s3_ivf.index"
)


print("\n==============================")
print("SAVING INDEX")
print("==============================")

print(
    "Saving to:",
    output_file
)


faiss.write_index(
    index,
    output_file
)


print("\nIndex saved successfully.")


# ============================================================
# 23. CHECK FILE
# ============================================================

if os.path.exists(output_file):

    file_size_gb = (
        os.path.getsize(output_file)
        / (1024 ** 3)
    )

    print("\n==============================")
    print("FILE INFORMATION")
    print("==============================")

    print(
        "File:",
        output_file
    )

    print(
        "Size:",
        f"{file_size_gb:.2f} GB"
    )

else:

    raise FileNotFoundError(
        "FAISS index file was not created."
    )


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print("\n============================================================")
print("FAISS INDEX READY")
print("============================================================")

print(
    "Total vectors:",
    f"{index.ntotal:,}"
)

print(
    "Dimension:",
    dimension
)

print(
    "nlist:",
    ivf_index.nlist
)

print(
    "nprobe:",
    ivf_index.nprobe
)

print(
    "Metric:",
    "INNER_PRODUCT"
)

print(
    "Index type:",
    "IndexIDMap2(IndexIVFFlat)"
)

print(
    "Saved at:",
    output_file
)

print("============================================================")

In [ ]:
dic={}

In [ ]:
# dic = {}

# k = 15

# def make_bucket(s1):

#     S1 = s1.copy()

#     S1["text"] = S1.apply(make_text, axis=1)

#     # Query embeddings
#     query_emb = model.encode(
#         S1["text"].tolist(),
#         normalize_embeddings=True,
#         show_progress_bar=True
#     )

#     query_emb = np.asarray(
#         query_emb,
#         dtype="float32"
#     )

#     # Search IVF
#     scores, faiss_indices = index.search(
#         query_emb,
#         k
#     )

#     for i in range(len(S1)):

#         score1 = scores[i].tolist()
#         index1 = faiss_indices[i].tolist()

#         dic[
#             S1.iloc[i]["entity_id"]
#         ] = [
#             index1,
#             score1
#         ]


# make_bucket(s1)

In [ ]:
make_bucket(s1)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

#tfid ka start

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd
import gc


# ============================================================
# TF-IDF VECTORISER TRAINING
# ============================================================

tfidf_sample_size = 500_000

rng = np.random.default_rng(42)


# ------------------------------------------------------------
# Name samples
# ------------------------------------------------------------

sample_s1 = min(
    tfidf_sample_size // 3,
    len(s1)
)

sample_s2 = min(
    tfidf_sample_size // 3,
    len(s2)
)

sample_s3 = min(
    tfidf_sample_size - sample_s1 - sample_s2,
    len(s3)
)


idx_s1 = rng.choice(
    len(s1),
    size=sample_s1,
    replace=False
)

idx_s2 = rng.choice(
    len(s2),
    size=sample_s2,
    replace=False
)

idx_s3 = rng.choice(
    len(s3),
    size=sample_s3,
    replace=False
)


# ============================================================
# NAME TF-IDF
# ============================================================

name_train_text = np.concatenate([

    s1["business_name"]
        .iloc[idx_s1]
        .fillna("")
        .astype(str)
        .values,

    s2["business_name"]
        .iloc[idx_s2]
        .fillna("")
        .astype(str)
        .values,

    s3["business_name"]
        .iloc[idx_s3]
        .fillna("")
        .astype(str)
        .values
])


print(
    "Fitting NAME TF-IDF..."
)

name_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    dtype=np.float32
)

name_vectorizer.fit(
    name_train_text
)

print(
    "Name TF-IDF vocabulary:",
    len(name_vectorizer.vocabulary_)
)


del name_train_text


# ============================================================
# ADDRESS TF-IDF
# ============================================================

address_train_text = np.concatenate([

    s1["business_address"]
        .iloc[idx_s1]
        .fillna("")
        .astype(str)
        .values,

    s2["business_address"]
        .iloc[idx_s2]
        .fillna("")
        .astype(str)
        .values,

    s3["business_address"]
        .iloc[idx_s3]
        .fillna("")
        .astype(str)
        .values
])


print(
    "Fitting ADDRESS TF-IDF..."
)

address_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    dtype=np.float32
)

address_vectorizer.fit(
    address_train_text
)

print(
    "Address TF-IDF vocabulary:",
    len(address_vectorizer.vocabulary_)
)


del address_train_text
del idx_s1
del idx_s2
del idx_s3

gc.collect()

print("TF-IDF vectorizers ready.")

In [ ]:
def tfidf_batch_similarity(
    query_texts,
    candidate_texts,
    vectorizer
):

    query_texts = [
        "" if pd.isna(x) else str(x)
        for x in query_texts
    ]

    candidate_texts = [
        "" if pd.isna(x) else str(x)
        for x in candidate_texts
    ]

    q = vectorizer.transform(
        query_texts
    )

    c = vectorizer.transform(
        candidate_texts
    )

    similarity = q.multiply(c).sum(
        axis=1
    )

    return np.asarray(
        similarity
    ).reshape(-1)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_similarity(text1, text2):
    text1 = "" if pd.isna(text1) else str(text1)
    text2 = "" if pd.isna(text2) else str(text2)
    if not text1.strip() or not text2.strip():
        return 0.0

    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 5)
    )

    X = vectorizer.fit_transform([text1, text2])

    return cosine_similarity(X[0], X[1])[0][0]


In [ ]:
def jaccard(a, b):

    if pd.isna(a):
        a = ""

    if pd.isna(b):
        b = ""

    a = str(a).lower()
    b = str(b).lower()

    A = set(a.split())
    B = set(b.split())

    if not A and not B:
        return 1.0

    if not A or not B:
        return 0.0

    return len(A & B) / len(A | B)

In [ ]:
from rapidfuzz.distance import Levenshtein
def Levenshtein1(name1,name2):
  distance = Levenshtein.distance(name1, name2)
  edit_sim = 1 - (
    distance / max(len(name1), len(name2))
  )
  return edit_sim

In [ ]:
def token_overlap1(a, b):
    A = set(a.lower().split())
    B = set(b.lower().split())

    if not A or not B:
        return 0.0

    return len(A & B) / min(len(A), len(B))

In [ ]:
def check(target, labels):

    if pd.isna(labels):
        return 0

    labels = str(labels)
    target = str(target)

    label_list = [
        x.strip().lower()
        for x in labels.split(",")
    ]

    return int(target.strip().lower() in label_list)

In [ ]:
def prepare_dataset(dic,s1):
  idname=[]
  faiss_scores = []
  tfid_name=[]
  tfid_add=[]
  jaccard_name=[]
  jaccard_address=[]
  levi=[]
  token_overlap=[]
  country_match=[]
  target_id=[]
  label=[]




  for i in range(len(s1)):

    currlist=dic[s1.iloc[i]["entity_id"]]
    real_label=truth.iloc[i]["matched_entity_ids"]
    name1=s1.iloc[i]["business_name"]
    add1=s1.iloc[i]["business_address"]
    country1=s1.iloc[i]["country"]
    score=currlist[1]
    index=currlist[0]
    p=0
    for j in index:
      id_tar=candidates.iloc[j]["entity_id"]
      target_id.append(id_tar)
      name=candidates.iloc[j]["business_name"]
      address=candidates.iloc[j]["business_address"]
      country=candidates.iloc[j]["country"]
      idname.append(s1.iloc[i]["entity_id"])
      faiss_scores.append(score[p])
      tfid_name.append(tfidf_similarity(name1,name))
      tfid_add.append(tfidf_similarity(add1,address))
      jaccard_name.append(jaccard(name1,name))
      jaccard_address.append(jaccard(add1,address))
      levi.append(Levenshtein1(name1,name))
      token_overlap.append(token_overlap1(name1,name))
      c=int(
    country1.lower() == country.lower()
    )
      country_match.append(c)
      label.append(check(name,real_label))


      p+=1


  df=pd.DataFrame({"entity_id":idname,"target_id":target_id,"faiss": faiss_scores,"tfid_name":tfid_name,"tfid_add":tfid_add,"jaccard_name":jaccard_name,"jaccard_address":jaccard_address,"levi":levi,"token_overlap":token_overlap,"country_match":country_match,"label":label})

  return df





In [ ]:
df=prepare_dataset(dic,s1)

In [ ]:
df

,entity_id,target_id,faiss,tfid_name,tfid_add,jaccard_name,jaccard_address,levi,token_overlap,country_match,label
0,S1-925783039,S2-415733261,0.602860,0.337069,0.068005,0.250,0.0,0.476190,0.50,1,0
1,S1-925783039,S3-171901473,0.583286,0.042616,0.032949,0.000,0.0,0.315789,0.00,1,0
2,S1-925783039,S2-69394310,0.555413,0.000000,0.051219,0.000,0.0,0.210526,0.00,1,0
3,S1-925783039,S3-671162755,0.555059,0.038141,0.062599,0.000,0.0,0.210526,0.00,1,0
4,S1-925783039,S3-973629095,0.552256,0.023631,0.093118,0.000,0.0,0.210526,0.00,1,0
...,...,...,...,...,...,...,...,...,...,...,...
1495,S1-746273151,S3-331103166,0.367012,0.044833,0.042187,0.000,0.0,0.258065,0.00,1,0
1496,S1-746273151,S2-414418413,0.359657,0.019406,0.043938,0.000,0.0,0.032258,0.00,1,0
1497,S1-746273151,S2-415733261,0.353686,0.000000,0.046641,0.000,0.0,0.129032,0.00,1,0
1498,S1-746273151,S2-69394310,0.349797,0.039621,0.070355,0.000,0.0,0.193548,0.00,1,0


In [ ]:
truth

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
...,...,...
95,S1-212509374,"S2-997766473,S2-598327313,S2-682745911,S3-6351..."
96,S1-359440945,"S2-683352932,S2-940771150,S3-75226047,S3-13720..."
97,S1-102422131,"S2-743086829,S3-778505908"
98,S1-862038247,S2-451274541


In [ ]:
check("","S2-681193310,S2-743505751,S3-775321672,S3-11291185,S3-860443364")

1

In [ ]:
dic = {}

k = 15

def make_bucket(s1):

    S1 = s1.copy()

    S1["text"] = S1.apply(make_text, axis=1)

    # Query embeddings
    query_emb = model.encode(
        S1["text"].tolist(),
        normalize_embeddings=True,
        show_progress_bar=True
    )

    query_emb = np.asarray(
        query_emb,
        dtype="float32"
    )

    # Search IVF
    scores, faiss_indices = index.search(
        query_emb,
        k
    )

    for i in range(len(S1)):

        score1 = scores[i].tolist()
        index1 = faiss_indices[i].tolist()

        dic[
            S1.iloc[i]["entity_id"]
        ] = [
            index1,
            score1
        ]


make_bucket(s1)

In [ ]:
def preparelist(text):
  return text.split(",")

In [ ]:
bucket_find()

In [ ]:
s2 = s2.set_index("entity_id")
s3 = s3.set_index("entity_id")

In [ ]:
# def bucket_find(
#     index,
#     s1,
#     s2,
#     s3,
#     truth,
#     emb1,
#     emb2,
#     emb3,
#     k_factor=2
# ):

#     # -----------------------------------------
#     # Entity ID as DataFrame index
#     # -----------------------------------------


#     # -----------------------------------------
#     # S2 / S3 offset in FAISS combined index
#     # -----------------------------------------

#     s2_offset = 0
#     s3_offset = len(s2)

#     dic = {}

#     for i in range(len(s1)):

#         # -----------------------------------------
#         # Ground truth IDs
#         # -----------------------------------------

#         gt_ids = preparelist(
#             truth.iloc[i]["matched_entity_ids"]
#         )

#         # -----------------------------------------
#         # Ground truth -> FAISS GLOBAL indices
#         # -----------------------------------------

#         gt_indices = []

#         for entity_id in gt_ids:

#             if entity_id in s2.index:

#                 local_idx = s2.index.get_loc(entity_id)

#                 global_idx = s2_offset + local_idx

#                 gt_indices.append(global_idx)

#             elif entity_id in s3.index:

#                 local_idx = s3.index.get_loc(entity_id)

#                 global_idx = s3_offset + local_idx

#                 gt_indices.append(global_idx)

#         # -----------------------------------------
#         # Query embedding
#         # -----------------------------------------

#         query_emb = np.asarray(
#             emb1[i],
#             dtype="float32"
#         ).reshape(1, -1)

#         # -----------------------------------------
#         # Positive similarities
#         # -----------------------------------------

#         positive_scores = []

#         for entity_id in gt_ids:

#             if entity_id in s2.index:

#                 idx = s2.index.get_loc(entity_id)

#                 sim = np.dot(
#                     query_emb[0],
#                     emb2[idx]
#                 )

#             elif entity_id in s3.index:

#                 idx = s3.index.get_loc(entity_id)

#                 sim = np.dot(
#                     query_emb[0],
#                     emb3[idx]
#                 )

#             else:
#                 continue

#             positive_scores.append(float(sim))

#         # -----------------------------------------
#         # FAISS search
#         # -----------------------------------------

#         k = min(
#             k_factor * len(gt_ids),
#             len(s2) + len(s3)
#         )

#         scores, indexes = index.search(
#             query_emb,
#             k
#         )

#         faiss_indices = indexes[0]
#         faiss_scores = scores[0]

#         # -----------------------------------------
#         # Negatives
#         # -----------------------------------------

#         negative_indices = []
#         negative_scores = []

#         gt_set = set(gt_indices)

#         for idx, score in zip(
#             faiss_indices,
#             faiss_scores
#         ):

#             if idx == -1:
#                 continue

#             if idx not in gt_set:

#                 negative_indices.append(int(idx))
#                 negative_scores.append(float(score))

#         # -----------------------------------------
#         # Combine
#         # -----------------------------------------

#         all_indices = gt_indices + negative_indices
#         all_scores = positive_scores + negative_scores

#         # -----------------------------------------
#         # Store
#         # -----------------------------------------

#         dic[
#             s1.iloc[i]["entity_id"]
#         ] = [
#             all_indices,
#             all_scores
#         ]

#     return dic

In [ ]:
def prepare_dataset_batch(
    index,
    s1_batch,
    s2,
    s3,
    truth_batch,
    emb1_batch,
    emb2,
    emb3,
    k_factor=2
):

    idname = []
    faiss_scores = []
    tfid_name = []
    tfid_add = []
    jaccard_name = []
    jaccard_address = []
    levi = []
    token_overlap = []
    country_match = []
    target_id = []
    label = []

    s3_offset = len(s2)

    for i in range(len(s1_batch)):

        gt_ids = preparelist(
            truth_batch.iloc[i]["matched_entity_ids"]
        )

        gt_indices = []

        for entity_id in gt_ids:

            if entity_id in s2.index:
                idx = s2.index.get_loc(entity_id)
                gt_indices.append(idx)

            elif entity_id in s3.index:
                idx = s3.index.get_loc(entity_id)
                gt_indices.append(s3_offset + idx)

        query_emb = np.asarray(
            emb1_batch[i],
            dtype="float32"
        ).reshape(1, -1)

        positive_scores = []

        for entity_id in gt_ids:

            if entity_id in s2.index:

                idx = s2.index.get_loc(entity_id)

                sim = np.dot(
                    query_emb[0],
                    emb2[idx]
                )

            elif entity_id in s3.index:

                idx = s3.index.get_loc(entity_id)

                sim = np.dot(
                    query_emb[0],
                    emb3[idx]
                )

            else:
                continue

            positive_scores.append(float(sim))

        # FAISS
        k = min(
            k_factor * len(gt_ids),
            len(s2) + len(s3)
        )

        scores, indexes = index.search(
            query_emb,
            k
        )

        negative_indices = []
        negative_scores = []

        gt_set = set(gt_indices)

        for idx, score in zip(indexes[0], scores[0]):

            if idx == -1:
                continue

            if idx not in gt_set:
                negative_indices.append(int(idx))
                negative_scores.append(float(score))

        all_indices = gt_indices + negative_indices
        all_scores = positive_scores + negative_scores

        # Features
        name1 = s1_batch.iloc[i]["business_name"]
        add1 = s1_batch.iloc[i]["business_address"]
        country1 = s1_batch.iloc[i]["country"]

        real_label = truth_batch.iloc[i]["matched_entity_ids"]
        s1_id = s1_batch.iloc[i]["entity_id"]

        for p, j in enumerate(all_indices):

            if j < len(s2):

                row = s2.iloc[j]

            else:

                row = s3.iloc[j - len(s2)]

            name = row["business_name"]
            address = row["business_address"]
            country = row["country"]

            idname.append(s1_id)
            target_id.append(row.name)
            faiss_scores.append(all_scores[p])

            tfid_name.append(
                tfidf_similarity(name1, name)
            )

            tfid_add.append(
                tfidf_similarity(add1, address)
            )

            jaccard_name.append(
                jaccard(name1, name)
            )

            jaccard_address.append(
                jaccard(add1, address)
            )

            levi.append(
                Levenshtein1(name1, name)
            )

            token_overlap.append(
                token_overlap1(name1, name)
            )

            country_match.append(
                int(
                    str(country1).lower()
                    == str(country).lower()
                )
            )

            label.append(
                check(name, real_label)
            )

    return pd.DataFrame({
        "entity_id": idname,
        "target_id": target_id,
        "faiss": faiss_scores,
        "tfid_name": tfid_name,
        "tfid_add": tfid_add,
        "jaccard_name": jaccard_name,
        "jaccard_address": jaccard_address,
        "levi": levi,
        "token_overlap": token_overlap,
        "country_match": country_match,
        "label": label
    })

In [ ]:
batch_size = 1000

for start in range(0, len(s1), batch_size):

    end = min(start + batch_size, len(s1))

    print(f"Processing {start}:{end}")

    s1_batch = s1.iloc[start:end]
    truth_batch = truth.iloc[start:end]

    emb1_batch = emb1[start:end]

    batch_df = prepare_dataset_batch(
        index,
        s1_batch,
        s2,
        s3,
        truth_batch,
        emb1_batch,
        emb2,
        emb3
    )

    # First batch -> create file
    if start == 0:

        batch_df.to_csv(
            "training_features.csv",
            index=False
        )

    # Next batches -> append
    else:

        batch_df.to_csv(
            "training_features.csv",
            mode="a",
            header=False,
            index=False
        )

    del batch_df

In [ ]:
def datatranformer(path,s1,s2,s3):
  df=pd.read_csv(path)
  source=[]
  target=[]
  labl=[]
  for i in range(len(df)):
    src=df.iloc[i]["entity_id"]
    tgt=df.iloc[i]["target_id"]
    label=df.iloc[i]["label"]
    source.append(s1.loc[src]["text"])
    if tgt in s2.index:
      target.append(s2.loc[tgt]["text"])
    elif tgt in s3.index:
      target.append(s3.loc[tgt]["text"])
    labl.append(label)

  df=pd.DataFrame({
      "source":source,
      "target":target,
      "label":labl
  })
  return df
#


In [ ]:
datatranformer("/content/training_features2.csv",s1,s2,s3)

,source,target,label
0,"Orelee's Barbershop 1795 Westchester Drive, Hi...","Orelee's Barbershop 1795 WESTCHESTER DRIVE, NC...",1
1,"Orelee's Barbershop 1795 Westchester Drive, Hi...","Orelee's Bárbershop 1795 Westchester Drive, HI...",1
2,"Orelee's Barbershop 1795 Westchester Drive, Hi...","Orelee's (Barbershop) #1795 Westchester Dr, No...",1
3,"Orelee's Barbershop 1795 Westchester Drive, Hi...","Orelee'S Services Westchester Dr, High Point, ...",1
4,"Orelee's Barbershop 1795 Westchester Drive, Hi...",Orelee's Barbershop Downtown 1799 WESTCHESTER ...,0
...,...,...,...
7137,Innovative Hospitality Private Limited 101/106...,Dynamic Hospitality Private Masalawaala Apartm...,0
7138,Innovative Hospitality Private Limited 101/106...,Dynamic Hospitality Private Ltd BHARGAVADUTT P...,0
7139,Innovative Hospitality Private Limited 101/106...,Innovative Hospitality Private Limited B 251 N...,0
7140,Innovative Hospitality Private Limited 101/106...,International One Hospitality Private Limited ...,0


In [ ]:
# ye code data csv file jab ho jaye usse tain and test ka tranformer ke liye data banvaya tha jisme dono ke text field ko le raha tha main sirf

import pandas as pd
import os
import time


def datatranformer(
    path,
    s1,
    s2,
    s3,
    output_path="training_transformed.csv",
    chunksize=100_000
):

    start_time = time.time()

    print("=" * 60)
    print("Starting data transformation...")
    print("=" * 60)

    # --------------------------------------------------
    # Create dictionaries for fast lookup
    # --------------------------------------------------

    print("Creating lookup dictionaries...")

    s1_text = s1["text"].to_dict()
    s2_text = s2["text"].to_dict()
    s3_text = s3["text"].to_dict()

    print(f"s1 lookup size: {len(s1_text):,}")
    print(f"s2 lookup size: {len(s2_text):,}")
    print(f"s3 lookup size: {len(s3_text):,}")

    # --------------------------------------------------
    # Remove previous output
    # --------------------------------------------------

    if os.path.exists(output_path):
        os.remove(output_path)
        print(f"Removed existing output: {output_path}")

    # --------------------------------------------------
    # Count total rows
    # --------------------------------------------------

    print("\nCounting input rows...")

    total_rows = sum(
        1 for _ in open(path, "r", encoding="utf-8")
    ) - 1

    print(f"Total input rows: {total_rows:,}")
    print(f"Batch size: {chunksize:,}")

    # --------------------------------------------------
    # Processing
    # --------------------------------------------------

    first_chunk = True
    processed = 0
    output_rows = 0
    batch_no = 0

    print("\nStarting processing...\n")

    for chunk in pd.read_csv(path, chunksize=chunksize):

        batch_no += 1

        source = []
        target = []
        label = []

        # ----------------------------------------------
        # Process current batch
        # ----------------------------------------------

        for src, tgt, lab in zip(
            chunk["entity_id"],
            chunk["target_id"],
            chunk["label"]
        ):

            # Source lookup
            src_text = s1_text.get(src)

            if src_text is None:
                continue

            # Target lookup
            tgt_text = s2_text.get(tgt)

            if tgt_text is None:
                tgt_text = s3_text.get(tgt)

            if tgt_text is None:
                continue

            source.append(src_text)
            target.append(tgt_text)
            label.append(lab)

        # ----------------------------------------------
        # Create output batch
        # ----------------------------------------------

        result = pd.DataFrame({
            "source": source,
            "target": target,
            "label": label
        })

        # ----------------------------------------------
        # Save batch
        # ----------------------------------------------

        result.to_csv(
            output_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False

        # ----------------------------------------------
        # Progress
        # ----------------------------------------------

        processed += len(chunk)
        output_rows += len(result)

        percentage = (processed / total_rows) * 100

        elapsed = time.time() - start_time

        print(
            f"Batch {batch_no:>4} | "
            f"Processed: {processed:,}/{total_rows:,} "
            f"({percentage:.2f}%) | "
            f"Output: {output_rows:,} | "
            f"Time: {elapsed/60:.2f} min"
        )

    # --------------------------------------------------
    # Finished
    # --------------------------------------------------

    total_time = time.time() - start_time

    print("\n" + "=" * 60)
    print("TRANSFORMATION COMPLETED")
    print("=" * 60)

    print(f"Input rows       : {total_rows:,}")
    print(f"Processed rows   : {processed:,}")
    print(f"Output rows      : {output_rows:,}")
    print(f"Output file      : {output_path}")
    print(f"Total time       : {total_time/60:.2f} minutes")
    print("=" * 60)

    return output_path

In [ ]:
output = datatranformer(
    "/content/training_features2.csv",
    s1,
    s2,
    s3,
    output_path="/content/training_transformed.csv",
    chunksize=100_000
)

Starting data transformation...
Creating lookup dictionaries...
s1 lookup size: 2,206,821
s2 lookup size: 5,034,616
s3 lookup size: 5,285,603

Counting input rows...
Total input rows: 7,142
Batch size: 100,000

Starting processing...

Batch    1 | Processed: 7,142/7,142 (100.00%) | Output: 7,142 | Time: 0.23 min

TRANSFORMATION COMPLETED
Input rows       : 7,142
Processed rows   : 7,142
Output rows      : 7,142
Output file      : /content/training_transformed.csv
Total time       : 0.23 minutes


In [ ]:
def prepare_dataset_batch(
    index,
    s1_batch,
    s2,
    s3,
    truth_batch,
    emb1_batch,
    emb2,
    emb3,
    k_factor=2
):

    idname = []
    faiss_scores = []
    tfid_name = []
    tfid_add = []
    jaccard_name = []
    jaccard_address = []
    levi = []
    token_overlap = []
    country_match = []
    target_id = []
    label = []

    s3_offset = len(s2)

    for i in range(len(s1_batch)):

        # =====================================================
        # Ground Truth IDs
        # =====================================================

        gt_ids = preparelist(
            truth_batch.iloc[i]["matched_entity_ids"]
        )

        # =====================================================
        # Query Embedding
        # =====================================================

        query_emb = np.asarray(
            emb1_batch[i],
            dtype="float32"
        ).reshape(1, -1)

        # =====================================================
        # Find valid ground-truth indices
        # =====================================================

        gt_indices = []

        for entity_id in gt_ids:

            if entity_id in s2.index:

                idx = s2.index.get_loc(entity_id)
                gt_indices.append(idx)

            elif entity_id in s3.index:

                idx = s3.index.get_loc(entity_id)
                gt_indices.append(
                    s3_offset + idx
                )

        # =====================================================
        # CASE 1: NO VALID GROUND TRUTH
        # =====================================================

        if not gt_indices:

            # Get top 2 nearest semantic candidates
            k = min(
                2,
                len(s2) + len(s3)
            )

            scores, indexes = index.search(
                query_emb,
                k
            )

            all_indices = []
            all_scores = []

            for idx, score in zip(
                indexes[0],
                scores[0]
            ):

                if idx == -1:
                    continue

                all_indices.append(
                    int(idx)
                )

                all_scores.append(
                    float(score)
                )

            # All candidates are negative
            all_labels = [
                0
            ] * len(all_indices)

        # =====================================================
        # CASE 2: GROUND TRUTH AVAILABLE
        # =====================================================

        else:

            positive_scores = []

            # -------------------------------------------------
            # Calculate similarity for positive GT records
            # -------------------------------------------------

            for entity_id in gt_ids:

                if entity_id in s2.index:

                    idx = s2.index.get_loc(entity_id)

                    sim = np.dot(
                        query_emb[0],
                        emb2[idx]
                    )

                elif entity_id in s3.index:

                    idx = s3.index.get_loc(entity_id)

                    sim = np.dot(
                        query_emb[0],
                        emb3[idx]
                    )

                else:
                    continue

                positive_scores.append(
                    float(sim)
                )

            # -------------------------------------------------
            # FAISS
            #
            # Example:
            # 2 GT -> k = 4
            # 3 GT -> k = 6
            # -------------------------------------------------

            k = min(
                k_factor * len(gt_indices),
                len(s2) + len(s3)
            )

            scores, indexes = index.search(
                query_emb,
                k
            )

            negative_indices = []
            negative_scores = []

            gt_set = set(gt_indices)

            # Number of negatives required
            # Same as number of positives
            op = len(gt_indices)

            for idx, score in zip(
                indexes[0],
                scores[0]
            ):

                if op <= 0:
                    break

                if idx == -1:
                    continue

                # Don't include ground-truth records
                if idx not in gt_set:

                    negative_indices.append(
                        int(idx)
                    )

                    negative_scores.append(
                        float(score)
                    )

                    op -= 1

            # -------------------------------------------------
            # Combine positives + negatives
            # -------------------------------------------------

            all_indices = (
                gt_indices
                +
                negative_indices
            )

            all_scores = (
                positive_scores
                +
                negative_scores
            )

            all_labels = (
                [1] * len(gt_indices)
                +
                [0] * len(negative_indices)
            )

        # =====================================================
        # Features
        # =====================================================

        name1 = s1_batch.iloc[i]["business_name"]
        add1 = s1_batch.iloc[i]["business_address"]
        country1 = s1_batch.iloc[i]["country"]

        # Handle NaN / missing values
        name1 = (
            ""
            if pd.isna(name1)
            else str(name1)
        )

        add1 = (
            ""
            if pd.isna(add1)
            else str(add1)
        )

        country1 = (
            ""
            if pd.isna(country1)
            else str(country1)
        )

        s1_id = s1_batch.iloc[i]["entity_id"]

        # =====================================================
        # Calculate features for candidates
        # =====================================================

        for p, j in enumerate(all_indices):

            # -------------------------------------------------
            # Get target row from S2 / S3
            # -------------------------------------------------

            if j < len(s2):

                row = s2.iloc[j]

            else:

                row = s3.iloc[
                    j - len(s2)
                ]

            name = row["business_name"]
            address = row["business_address"]
            country = row["country"]

            # Handle NaN / missing values
            name = (
                ""
                if pd.isna(name)
                else str(name)
            )

            address = (
                ""
                if pd.isna(address)
                else str(address)
            )

            country = (
                ""
                if pd.isna(country)
                else str(country)
            )

            # -------------------------------------------------
            # Basic information
            # -------------------------------------------------

            idname.append(
                s1_id
            )

            target_id.append(
                row.name
            )

            faiss_scores.append(
                all_scores[p]
            )

            # -------------------------------------------------
            # Name TF-IDF
            # -------------------------------------------------

            tfid_name.append(
                tfidf_similarity(
                    name1,
                    name
                )
            )

            # -------------------------------------------------
            # Address TF-IDF
            # -------------------------------------------------

            tfid_add.append(
                tfidf_similarity(
                    add1,
                    address
                )
            )

            # -------------------------------------------------
            # Name Jaccard
            # -------------------------------------------------

            jaccard_name.append(
                jaccard(
                    name1,
                    name
                )
            )

            # -------------------------------------------------
            # Address Jaccard
            # -------------------------------------------------

            jaccard_address.append(
                jaccard(
                    add1,
                    address
                )
            )

            # -------------------------------------------------
            # Levenshtein
            # -------------------------------------------------

            levi.append(
                Levenshtein1(
                    name1,
                    name
                )
            )

            # -------------------------------------------------
            # Token overlap
            # -------------------------------------------------

            token_overlap.append(
                token_overlap1(
                    name1,
                    name
                )
            )

            # -------------------------------------------------
            # Country match
            # -------------------------------------------------

            country_match.append(
                int(
                    country1.lower()
                    ==
                    country.lower()
                )
            )

            # -------------------------------------------------
            # Label
            # -------------------------------------------------

            label.append(
                all_labels[p]
            )

    # =========================================================
    # Return batch DataFrame
    # =========================================================

    return pd.DataFrame({

        "entity_id": idname,

        "target_id": target_id,

        "faiss": faiss_scores,

        "tfid_name": tfid_name,

        "tfid_add": tfid_add,

        "jaccard_name": jaccard_name,

        "jaccard_address": jaccard_address,

        "levi": levi,

        "token_overlap": token_overlap,

        "country_match": country_match,

        "label": label
    })

In [ ]:
batch_size = 1000

for start in range(0, len(s1), batch_size):

    end = min(
        start + batch_size,
        len(s1)
    )

    print(f"Processing {start}:{end}")

    s1_batch = s1.iloc[start:end]

    truth_batch = truth_aligned.iloc[start:end]

    emb1_batch = emb1[start:end]

    batch_df = prepare_dataset_batch(
        index,
        s1_batch,
        s2,
        s3,
        truth_batch,
        emb1_batch,
        emb2,
        emb3
    )

    if start == 0:

        batch_df.to_csv(
            "training_features4.csv",
            index=False
        )

    else:

        batch_df.to_csv(
            "training_features4.csv",
            mode="a",
            header=False,
            index=False
        )

    del batch_df

In [ ]:
# testing data
def prepare_test_bucket_batch(
    index,
    s1_batch,
    s2,
    s3,
    emb1_batch,
    k=12
):

    idname = []
    target_id = []
    faiss_scores = []

    tfid_name = []
    tfid_add = []
    jaccard_name = []
    jaccard_address = []
    levi = []
    token_overlap = []
    country_match = []

    query_emb = np.asarray(
        emb1_batch,
        dtype="float32"
    )

    # FAISS search for ONLY this batch
    scores, faiss_indices = index.search(
        query_emb,
        k
    )

    for i in range(len(s1_batch)):

        name1 = s1_batch.iloc[i]["business_name"]
        add1 = s1_batch.iloc[i]["business_address"]
        country1 = s1_batch.iloc[i]["country"]

        s1_id = s1_batch.iloc[i]["entity_id"]

        # Top-12 candidates for this S1
        candidate_indices = faiss_indices[i]
        candidate_scores = scores[i]

        for p, j in enumerate(candidate_indices):

            if j == -1:
                continue

            # -----------------------------
            # S2 / S3 identify
            # -----------------------------

            if j < len(s2):

                row = s2.iloc[j]

            else:

                s3_idx = j - len(s2)

                row = s3.iloc[s3_idx]

            # -----------------------------
            # Candidate data
            # -----------------------------

            name = row["business_name"]
            address = row["business_address"]
            country = row["country"]

            # -----------------------------
            # IDs
            # -----------------------------

            idname.append(s1_id)

            target_id.append(
                row["entity_id"]
            )

            # -----------------------------
            # FAISS
            # -----------------------------

            faiss_scores.append(
                float(candidate_scores[p])
            )

            # -----------------------------
            # Features
            # -----------------------------

            tfid_name.append(
                tfidf_similarity(
                    name1,
                    name
                )
            )

            tfid_add.append(
                tfidf_similarity(
                    add1,
                    address
                )
            )

            jaccard_name.append(
                jaccard(
                    name1,
                    name
                )
            )

            jaccard_address.append(
                jaccard(
                    add1,
                    address
                )
            )

            levi.append(
                Levenshtein1(
                    name1,
                    name
                )
            )

            token_overlap.append(
                token_overlap1(
                    name1,
                    name
                )
            )

            country_match.append(
                int(
                    str(country1).lower()
                    ==
                    str(country).lower()
                )
            )

    return pd.DataFrame({

        "entity_id": idname,

        "target_id": target_id,

        "faiss": faiss_scores,

        "tfid_name": tfid_name,

        "tfid_add": tfid_add,

        "jaccard_name": jaccard_name,

        "jaccard_address": jaccard_address,

        "levi": levi,

        "token_overlap": token_overlap,

        "country_match": country_match

    })

In [ ]:
batch_size = 1000

output_file = "testing_features.csv"

for start in range(0, len(s1), batch_size):

    end = min(
        start + batch_size,
        len(s1)
    )

    print(f"Processing {start}:{end}")

    # -----------------------------
    # S1 batch
    # -----------------------------

    s1_batch = s1.iloc[start:end]

    # -----------------------------
    # Corresponding embeddings
    # -----------------------------

    emb1_batch = emb1[start:end]

    # -----------------------------
    # Prepare testing bucket
    # -----------------------------

    batch_df = prepare_test_bucket_batch(
        index=index,
        s1_batch=s1_batch,
        s2=s2,
        s3=s3,
        emb1_batch=emb1_batch,
        k=12
    )

    # -----------------------------
    # Save
    # -----------------------------

    if start == 0:

        batch_df.to_csv(
            output_file,
            index=False
        )

    else:

        batch_df.to_csv(
            output_file,
            mode="a",
            header=False,
            index=False
        )

    # -----------------------------
    # Free memory
    # -----------------------------

    del batch_df

    print(f"Saved {start}:{end}")

In [ ]:
s1.index

'S1-925783039'

In [ ]:
len(s1)

2206821

# model training

In [ ]:
import pandas as pd

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df=pd.read_csv("/content/drive/MyDrive/amazonml/training_features4.csv")

In [ ]:
df1=df.drop(columns=["entity_id","target_id"])

In [ ]:
X=df1.drop(columns=["label"])
y=df1["label"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
  X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [ ]:
from xgboost import XGBClassifier

In [ ]:
from xgboost.callback import TrainingCallback

class ProgressBar(TrainingCallback):

    def __init__(self, total):
        self.total = total

    def after_iteration(self, model, epoch, evals_log):
        percent = (epoch + 1) / self.total * 100

        bar_length = 30
        filled = int(bar_length * (epoch + 1) / self.total)
        bar = "█" * filled + "-" * (bar_length - filled)

        print(
            f"\rTraining: |{bar}| {percent:5.1f}%",
            end=""
        )

        return False

    def after_training(self, model):
        print()
        return model

In [ ]:
from xgboost import XGBClassifier
from xgboost.callback import TrainingCallback


class ProgressBar(TrainingCallback):

    def __init__(self, total):
        self.total = total

    def after_iteration(self, model, epoch, evals_log):
        percent = (epoch + 1) / self.total * 100

        bar_length = 30
        filled = int(bar_length * (epoch + 1) / self.total)

        bar = "█" * filled + "-" * (bar_length - filled)

        print(
            f"\rTraining: |{bar}| {percent:5.1f}%",
            end=""
        )

        return False

    def after_training(self, model):
        print()
        return model


n_estimators = 1000

model = XGBClassifier(
    n_estimators=n_estimators,
    max_depth=50,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,

    # GPU
    device="cuda",
    n_jobs=-1,
    t
    # Progress bar
    callbacks=[ProgressBar(n_estimators)]
)


model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

Training: |██████████████████████████████| 100.0%


XGBClassifier(base_score=None, booster=None,
              callbacks=[<__main__.ProgressBar object at 0x7d7e3d9e0830>],
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=50, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
y_pred = model.predict(X_test)

print(y_pred)

[0 0 0 ... 0 1 1]


In [ ]:
y_pred

array([0, 0, 0, ..., 0, 1, 1])

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.9336136376637492
              precision    recall  f1-score   support

           0       0.95      0.92      0.93    337440
           1       0.92      0.95      0.93    326838

    accuracy                           0.93    664278
   macro avg       0.93      0.93      0.93    664278
weighted avg       0.93      0.93      0.93    664278



# Transformer

In [ ]:
%%writefile train.py
import torch
import numpy as np
import pandas as pd

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)


# ============================================================
# 1. CONFIG
# ============================================================

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

TRAIN_DATASET_PATH = "./train_dataset"
VAL_DATASET_PATH = "./val_dataset"

OUTPUT_DIR = "./mpnet-finetuned"


# ============================================================
# 2. GPU INFORMATION
# ============================================================

print("=" * 60)
print("GPU INFORMATION")
print("=" * 60)

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}: "
            f"{torch.cuda.get_device_name(i)}"
        )

print("=" * 60)


# ============================================================
# 3. LOAD CSV
# ============================================================

CSV_PATH = "/kaggle/input/datasets/amaynagar/amazon-ml-transformer/training_transformed.csv"

df = pd.read_csv(CSV_PATH)

print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

print(df.head())
print("\nLabel distribution:")
print(df["label"].value_counts())


# ============================================================
# 4. TRAIN / VALIDATION SPLIT
# ============================================================

train_df, val_df = train_test_split(
    df,
    test_size=0.25,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

print("\nTrain:", len(train_dataset))
print("Validation:", len(val_dataset))

# ============================================================
# 5. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 5. MODEL
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)


# ============================================================
# 6. TOKENIZATION
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["source"],
        examples["target"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


print("Tokenizing train dataset...")

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing train"
)


print("Tokenizing validation dataset...")

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing validation"
)


# ============================================================
# 7. REMOVE RAW TEXT
# ============================================================

train_dataset = train_dataset.remove_columns(
    ["source", "target"]
)

val_dataset = val_dataset.remove_columns(
    ["source", "target"]
)


# ============================================================
# 8. TORCH FORMAT
# ============================================================

train_dataset.set_format("torch")
val_dataset.set_format("torch")


# ============================================================
# 9. METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ============================================================
# 10. TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    output_dir=OUTPUT_DIR,

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    eval_strategy="epoch",

    # --------------------------------------------------------
    # Saving
    # --------------------------------------------------------

    save_strategy="epoch",

    save_total_limit=2,

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    # --------------------------------------------------------
    # Best model
    # --------------------------------------------------------

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    logging_steps=10,

    logging_strategy="steps",

    report_to="none",

    # --------------------------------------------------------
    # Mixed precision
    # --------------------------------------------------------

    fp16=True,

    # --------------------------------------------------------
    # Dataloader
    # --------------------------------------------------------

    dataloader_num_workers=2,

    # --------------------------------------------------------
    # Distributed training
    # --------------------------------------------------------

    ddp_find_unused_parameters=False
)


# ============================================================
# 11. TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)


# ============================================================
# 12. TRAIN
# ============================================================

print("\nStarting training...\n")

train_result = trainer.train()


# ============================================================
# 13. FINAL EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

metrics = trainer.evaluate()

for key, value in metrics.items():

    if isinstance(value, float):

        print(
            f"{key}: {value:.4f}"
        )

    else:

        print(
            f"{key}: {value}"
        )


# ============================================================
# 14. SAVE FINAL MODEL
# ============================================================

FINAL_MODEL_DIR = "./mpnet-same-different"

print("\nSaving final model...")

trainer.save_model(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)


# ============================================================
# 15. SAVE TRAINING METRICS
# ============================================================

trainer.save_metrics(
    "train",
    train_result.metrics
)

trainer.save_metrics(
    "eval",
    metrics
)


print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(
    "Final model saved at:",
    FINAL_MODEL_DIR
)

In [ ]:
!torchrun --nproc_per_node=2 train.py